#### Inisialiate Spark

In [1]:
import sys, glob
spark_lib = '/opt/spark/python/lib'
for z in glob.glob(spark_lib + '/*.zip'):
    sys.path.insert(0, z)
sys.path.insert(0, '/opt/spark/python')

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, abs as spark_abs, round as spark_round, lit, median

spark = SparkSession.builder \
    .master("spark://spark-master:7077") \
    .appName("SilverCleaningNB") \
    .getOrCreate()

df_train = spark.read.csv("hdfs://namenode:8020/data/bronze/home_credit/raw/application_train.csv", header=True, inferSchema=True)
print("Train count:", df_train.count())
df_train.show(5, truncate=False)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/10 14:08:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Train count: 307511


26/08/10 14:08:58 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+----------+------+------------------+-----------+------------+---------------+------------+----------------+----------+-----------+---------------+---------------+----------------+-----------------------------+--------------------+-----------------+--------------------------+----------+-------------+-----------------+---------------+-----------+----------+--------------+---------------+----------------+----------+----------+---------------+---------------+--------------------+---------------------------+--------------------------+-----------------------+--------------------------+--------------------------+---------------------------+----------------------+----------------------+-----------------------+----------------------+-------------------+------------------+-------------------+--------------+----------------+---------------------------+------------------+--------------+-------------+-------------+-------------+-------------+------------+--------------------+--------------+------

#### Imputasi

In [2]:
def fill_missing_with_median(df, col_name):
    med = df.select(median(col(col_name))).collect()[0][0]
    if med is not None:
        df = df.withColumn(col_name, when(col(col_name).isNull(), lit(med)).otherwise(col(col_name)))
    return df

def fill_missing_with_mode(df, col_name, default="Unknown"):
    mode_row = df.groupBy(col_name).count().orderBy("count", ascending=False).first()
    mode_val = mode_row[col_name] if mode_row and mode_row[col_name] is not None else default
    df = df.withColumn(col_name, when(col(col_name).isNull(), lit(mode_val)).otherwise(col(col_name)))
    return df

#### Transformasi

In [3]:

df_train = df_train.withColumn("AGE_YEARS", spark_round(spark_abs(col("DAYS_BIRTH")) / 365.25, 2))
df_train = df_train.drop("DAYS_BIRTH")

df_train = df_train.withColumn("FLAG_UNEMPLOYED", when(col("DAYS_EMPLOYED") == 365243, 1).otherwise(0))
df_train = df_train.withColumn("YEARS_EMPLOYED", 
                   when(col("DAYS_EMPLOYED") == 365243, 0.0)
                   .otherwise(spark_round(spark_abs(col("DAYS_EMPLOYED")) / 365.25, 2)))
df_train = df_train.drop("DAYS_EMPLOYED")

for flag_col in ["FLAG_OWN_CAR", "FLAG_OWN_REALTY"]:
    if flag_col in df_train.columns:
        df_train = df_train.withColumn(flag_col, when(col(flag_col) == "Y", 1).otherwise(0).cast("int"))

# Lihat hasil sementara
df_train.select("AGE_YEARS", "YEARS_EMPLOYED", "FLAG_UNEMPLOYED", "FLAG_OWN_CAR", "FLAG_OWN_REALTY").show(5)

+---------+--------------+---------------+------------+---------------+
|AGE_YEARS|YEARS_EMPLOYED|FLAG_UNEMPLOYED|FLAG_OWN_CAR|FLAG_OWN_REALTY|
+---------+--------------+---------------+------------+---------------+
|     25.9|          1.74|              0|           0|              1|
|     45.9|          3.25|              0|           0|              0|
|    52.15|          0.62|              0|           1|              1|
|    52.03|          8.32|              0|           0|              1|
|    54.57|          8.32|              0|           0|              1|
+---------+--------------+---------------+------------+---------------+
only showing top 5 rows



#### Imputasi Missing Value

In [4]:
from pyspark.sql.functions import col, when, count, lit
from pyspark.sql.types import DoubleType

def fill_missing_with_median_fast(df, col_name):
    
    q = df.select(col_name).approxQuantile(col_name, [0.5], 0.01)
    med = q[0] if q else None
    if med is not None:
        df = df.withColumn(col_name, when(col(col_name).isNull(), lit(med)).otherwise(col(col_name)))
    return df

def fill_missing_with_mode(df, col_name, default="Unknown"):
    mode_row = df.groupBy(col_name).count().orderBy("count", ascending=False).first()
    mode_val = mode_row[col_name] if mode_row and mode_row[col_name] is not None else default
    df = df.withColumn(col_name, when(col(col_name).isNull(), lit(mode_val)).otherwise(col(col_name)))
    return df

numeric_cols = [c for c, t in df_train.dtypes if t in ["double", "int", "float"] and c not in ("TARGET", "SK_ID_CURR")]
for nc in numeric_cols:
    if df_train.filter(col(nc).isNull()).count() > 0:
        df_train = fill_missing_with_median_fast(df_train, nc)


cat_cols = [c for c, t in df_train.dtypes if t == "string"]
for cc in cat_cols:
    if df_train.filter(col(cc).isNull()).count() > 0:
        df_train = fill_missing_with_mode(df_train, cc)


nulls_after = df_train.select([count(when(col(c).isNull(), c)).alias(c) for c in numeric_cols[:5]])
nulls_after.show()

26/08/10 14:11:26 ERROR TaskSchedulerImpl: Lost executor 0 on 172.18.0.8: worker lost: Not receiving heartbeat for 60 seconds
26/08/10 14:11:27 ERROR TaskSchedulerImpl: Ignoring update with state FINISHED for TID 27 because its task set is gone (this is likely the result of receiving duplicate task finished status updates) or its executor has been marked as failed.
26/08/10 14:11:27 WARN StandaloneSchedulerBackend$StandaloneDriverEndpoint: Ignored task status update (27 state FINISHED) from unknown executor with ID 0


+------------+---------------+------------+----------------+----------+
|FLAG_OWN_CAR|FLAG_OWN_REALTY|CNT_CHILDREN|AMT_INCOME_TOTAL|AMT_CREDIT|
+------------+---------------+------------+----------------+----------+
|           0|              0|           0|               0|         0|
+------------+---------------+------------+----------------+----------+



#### Dedpulikasi & Save Stagging

In [5]:
count_before = df_train.count()
df_train = df_train.dropDuplicates(["SK_ID_CURR"])
count_after = df_train.count()
print(f"Sebelum dedup: {count_before}, sesudah: {count_after}")

df_train.write.mode("overwrite").parquet("hdfs://namenode:8020/data/silver/staging/application_train_clean")
print("Train clean saved.")

Sebelum dedup: 307511, sesudah: 307511


Train clean saved.


#### Aplication_test.csv

In [6]:

df_test = spark.read.csv("hdfs://namenode:8020/data/bronze/home_credit/raw/application_test.csv", header=True, inferSchema=True)
print("Test count:", df_test.count())

df_test = df_test.withColumn("AGE_YEARS", spark_round(spark_abs(col("DAYS_BIRTH")) / 365.25, 2))
df_test = df_test.drop("DAYS_BIRTH")
df_test = df_test.withColumn("FLAG_UNEMPLOYED", when(col("DAYS_EMPLOYED") == 365243, 1).otherwise(0))
df_test = df_test.withColumn("YEARS_EMPLOYED", 
                 when(col("DAYS_EMPLOYED") == 365243, 0.0)
                 .otherwise(spark_round(spark_abs(col("DAYS_EMPLOYED")) / 365.25, 2)))
df_test = df_test.drop("DAYS_EMPLOYED")

for flag_col in ["FLAG_OWN_CAR", "FLAG_OWN_REALTY"]:
    if flag_col in df_test.columns:
        df_test = df_test.withColumn(flag_col, when(col(flag_col) == "Y", 1).otherwise(0).cast("int"))

test_num_cols = [c for c, t in df_test.dtypes if t in ["double", "int", "float"] and c not in ("TARGET", "SK_ID_CURR")]
for nc in test_num_cols:
    if df_test.filter(col(nc).isNull()).count() > 0:
        df_test = fill_missing_with_median(df_test, nc)

test_cat_cols = [c for c, t in df_test.dtypes if t == "string"]
for cc in test_cat_cols:
    if df_test.filter(col(cc).isNull()).count() > 0:
        df_test = fill_missing_with_mode(df_test, cc)

df_test = df_test.dropDuplicates(["SK_ID_CURR"])

df_test.write.mode("overwrite").parquet("hdfs://namenode:8020/data/silver/staging/application_test_clean")
print("Test clean saved.")

Test count: 48744


Test clean saved.


#### Verif

In [7]:
df_train_check = spark.read.parquet("hdfs://namenode:8020/data/silver/staging/application_train_clean")
print("Train clean count:", df_train_check.count())
df_train_check.select("SK_ID_CURR", "TARGET", "AGE_YEARS", "YEARS_EMPLOYED", "FLAG_UNEMPLOYED").show(5)

df_test_check = spark.read.parquet("hdfs://namenode:8020/data/silver/staging/application_test_clean")
print("Test clean count:", df_test_check.count())
df_test_check.select("SK_ID_CURR", "AGE_YEARS", "YEARS_EMPLOYED", "FLAG_UNEMPLOYED").show(5)

Train clean count: 307511
+----------+------+---------+--------------+---------------+
|SK_ID_CURR|TARGET|AGE_YEARS|YEARS_EMPLOYED|FLAG_UNEMPLOYED|
+----------+------+---------+--------------+---------------+
|    100002|     1|     25.9|          1.74|              0|
|    100004|     0|    52.15|          0.62|              0|
|    100006|     0|    52.03|          8.32|              0|
|    100009|     0|    37.72|          8.57|              0|
|    100012|     0|    39.61|          5.53|              0|
+----------+------+---------+--------------+---------------+
only showing top 5 rows

Test clean count: 48744
+----------+---------+--------------+---------------+
|SK_ID_CURR|AGE_YEARS|YEARS_EMPLOYED|FLAG_UNEMPLOYED|
+----------+---------+--------------+---------------+
|    100013|    54.86|         12.21|              0|
|    100028|    38.26|          5.11|              0|
|    100038|     35.7|           6.0|              0|
|    100057|    45.68|          7.06|              0

### Bureau.csv

In [9]:
df_bureau = spark.read.csv("hdfs://namenode:8020/data/bronze/home_credit/raw/bureau.csv", header=True, inferSchema=True)
df_bureau.createOrReplaceTempView("bureau_raw")

spark.sql("""
    CREATE OR REPLACE TEMP VIEW bureau_clean AS
    SELECT DISTINCT
        SK_ID_CURR,
        SK_ID_BUREAU,
        CREDIT_ACTIVE,
        CREDIT_CURRENCY,
        COALESCE(DAYS_CREDIT, 0) AS DAYS_CREDIT,
        COALESCE(CREDIT_DAY_OVERDUE, 0) AS CREDIT_DAY_OVERDUE,
        COALESCE(DAYS_CREDIT_ENDDATE, 0) AS DAYS_CREDIT_ENDDATE,
        COALESCE(DAYS_ENDDATE_FACT, 0) AS DAYS_ENDDATE_FACT,
        COALESCE(AMT_CREDIT_MAX_OVERDUE, 0) AS AMT_CREDIT_MAX_OVERDUE,
        COALESCE(CNT_CREDIT_PROLONG, 0) AS CNT_CREDIT_PROLONG,
        COALESCE(AMT_CREDIT_SUM, 0) AS AMT_CREDIT_SUM,
        COALESCE(AMT_CREDIT_SUM_DEBT, 0) AS AMT_CREDIT_SUM_DEBT,
        COALESCE(AMT_CREDIT_SUM_LIMIT, 0) AS AMT_CREDIT_SUM_LIMIT,
        COALESCE(AMT_CREDIT_SUM_OVERDUE, 0) AS AMT_CREDIT_SUM_OVERDUE,
        COALESCE(CREDIT_TYPE, 'Unknown') AS CREDIT_TYPE,
        COALESCE(DAYS_CREDIT_UPDATE, 0) AS DAYS_CREDIT_UPDATE,
        COALESCE(AMT_ANNUITY, 0) AS AMT_ANNUITY
    FROM bureau_raw
""")

spark.table("bureau_clean").write.mode("overwrite").parquet("hdfs://namenode:8020/data/silver/staging/bureau_clean")

### Bureau_balance.csv

In [10]:
df_bb = spark.read.csv("hdfs://namenode:8020/data/bronze/home_credit/raw/bureau_balance.csv", header=True, inferSchema=True)
df_bb.createOrReplaceTempView("bb_raw")

spark.sql("""
    CREATE OR REPLACE TEMP VIEW bb_clean AS
    SELECT DISTINCT
        SK_ID_BUREAU,
        MONTHS_BALANCE,
        COALESCE(STATUS, 'C') AS STATUS
    FROM bb_raw
""")

spark.table("bb_clean").write.mode("overwrite").parquet("hdfs://namenode:8020/data/silver/staging/bureau_balance_clean")

26/08/10 14:18:47 ERROR TaskSchedulerImpl: Lost executor 1 on 172.18.0.8: Command exited with code 137
26/08/10 14:18:47 WARN TaskSetManager: Lost task 3.0 in stage 1042.0 (TID 1111) (172.18.0.8 executor 1): ExecutorLostFailure (executor 1 exited caused by one of the running tasks) Reason: Command exited with code 137
26/08/10 14:18:47 WARN TaskSetManager: Lost task 2.0 in stage 1042.0 (TID 1110) (172.18.0.8 executor 1): ExecutorLostFailure (executor 1 exited caused by one of the running tasks) Reason: Command exited with code 137
26/08/10 14:18:52 WARN TaskSetManager: Lost task 3.1 in stage 1042.0 (TID 1113) (172.18.0.8 executor 2): FetchFailed(null, shuffleId=323, mapIndex=-1, mapId=-1, reduceId=138, message=
org.apache.spark.shuffle.MetadataFetchFailedException: Missing an output location for shuffle 323 partition 138
	at org.apache.spark.MapOutputTracker$.validateStatus(MapOutputTracker.scala:1739)
	at org.apache.spark.MapOutputTracker$.$anonfun$convertMapStatuses$11(MapOutputTrack

### Previous_application.csv

In [11]:
df_prev = spark.read.csv("hdfs://namenode:8020/data/bronze/home_credit/raw/previous_application.csv", header=True, inferSchema=True)
df_prev.createOrReplaceTempView("prev_raw")

spark.sql("""
    CREATE OR REPLACE TEMP VIEW prev_clean AS
    SELECT DISTINCT
        SK_ID_PREV,
        SK_ID_CURR,
        COALESCE(NAME_CONTRACT_TYPE, 'Unknown') AS NAME_CONTRACT_TYPE,
        COALESCE(AMT_ANNUITY, 0) AS AMT_ANNUITY,
        COALESCE(AMT_APPLICATION, 0) AS AMT_APPLICATION,
        COALESCE(AMT_CREDIT, 0) AS AMT_CREDIT,
        COALESCE(AMT_DOWN_PAYMENT, 0) AS AMT_DOWN_PAYMENT,
        COALESCE(AMT_GOODS_PRICE, 0) AS AMT_GOODS_PRICE,
        COALESCE(WEEKDAY_APPR_PROCESS_START, 'Unknown') AS WEEKDAY_APPR_PROCESS_START,
        COALESCE(HOUR_APPR_PROCESS_START, 0) AS HOUR_APPR_PROCESS_START,
        COALESCE(FLAG_LAST_APPL_PER_CONTRACT, 'N') AS FLAG_LAST_APPL_PER_CONTRACT,
        COALESCE(NFLAG_LAST_APPL_IN_DAY, 0) AS NFLAG_LAST_APPL_IN_DAY,
        COALESCE(RATE_DOWN_PAYMENT, 0) AS RATE_DOWN_PAYMENT,
        COALESCE(RATE_INTEREST_PRIMARY, 0) AS RATE_INTEREST_PRIMARY,
        COALESCE(RATE_INTEREST_PRIVILEGED, 0) AS RATE_INTEREST_PRIVILEGED,
        COALESCE(NAME_CASH_LOAN_PURPOSE, 'Unknown') AS NAME_CASH_LOAN_PURPOSE,
        COALESCE(NAME_CONTRACT_STATUS, 'Unknown') AS NAME_CONTRACT_STATUS,
        COALESCE(DAYS_DECISION, 0) AS DAYS_DECISION,
        COALESCE(NAME_PAYMENT_TYPE, 'Unknown') AS NAME_PAYMENT_TYPE,
        COALESCE(CODE_REJECT_REASON, 'Unknown') AS CODE_REJECT_REASON,
        COALESCE(NAME_TYPE_SUITE, 'Unknown') AS NAME_TYPE_SUITE,
        COALESCE(NAME_CLIENT_TYPE, 'Unknown') AS NAME_CLIENT_TYPE,
        COALESCE(NAME_GOODS_CATEGORY, 'Unknown') AS NAME_GOODS_CATEGORY,
        COALESCE(NAME_PORTFOLIO, 'Unknown') AS NAME_PORTFOLIO,
        COALESCE(NAME_PRODUCT_TYPE, 'Unknown') AS NAME_PRODUCT_TYPE,
        COALESCE(CHANNEL_TYPE, 'Unknown') AS CHANNEL_TYPE,
        COALESCE(SELLERPLACE_AREA, 0) AS SELLERPLACE_AREA,
        COALESCE(NAME_SELLER_INDUSTRY, 'Unknown') AS NAME_SELLER_INDUSTRY,
        COALESCE(CNT_PAYMENT, 0) AS CNT_PAYMENT,
        COALESCE(NAME_YIELD_GROUP, 'Unknown') AS NAME_YIELD_GROUP,
        COALESCE(PRODUCT_COMBINATION, 'Unknown') AS PRODUCT_COMBINATION,
        COALESCE(DAYS_FIRST_DRAWING, 0) AS DAYS_FIRST_DRAWING,
        COALESCE(DAYS_FIRST_DUE, 0) AS DAYS_FIRST_DUE,
        COALESCE(DAYS_LAST_DUE_1ST_VERSION, 0) AS DAYS_LAST_DUE_1ST_VERSION,
        COALESCE(DAYS_LAST_DUE, 0) AS DAYS_LAST_DUE,
        COALESCE(DAYS_TERMINATION, 0) AS DAYS_TERMINATION,
        COALESCE(NFLAG_INSURED_ON_APPROVAL, 0) AS NFLAG_INSURED_ON_APPROVAL
    FROM prev_raw
""")

spark.table("prev_clean").write.mode("overwrite").parquet("hdfs://namenode:8020/data/silver/staging/previous_application_clean")

### POS_CASH_Balance.csv

In [12]:
df_pos = spark.read.csv("hdfs://namenode:8020/data/bronze/home_credit/raw/POS_CASH_balance.csv", header=True, inferSchema=True)
df_pos.createOrReplaceTempView("pos_raw")

spark.sql("""
    CREATE OR REPLACE TEMP VIEW pos_clean AS
    SELECT DISTINCT
        SK_ID_PREV,
        SK_ID_CURR,
        MONTHS_BALANCE,
        COALESCE(CNT_INSTALMENT, 0) AS CNT_INSTALMENT,
        COALESCE(CNT_INSTALMENT_FUTURE, 0) AS CNT_INSTALMENT_FUTURE,
        COALESCE(NAME_CONTRACT_STATUS, 'Unknown') AS NAME_CONTRACT_STATUS,
        COALESCE(SK_DPD, 0) AS SK_DPD,
        COALESCE(SK_DPD_DEF, 0) AS SK_DPD_DEF
    FROM pos_raw
""")

spark.table("pos_clean").write.mode("overwrite").parquet("hdfs://namenode:8020/data/silver/staging/POS_CASH_balance_clean")

26/08/10 14:21:58 ERROR TaskSchedulerImpl: Lost executor 2 on 172.18.0.8: Command exited with code 137
26/08/10 14:21:58 WARN TaskSetManager: Lost task 2.0 in stage 1052.0 (TID 1144) (172.18.0.8 executor 2): ExecutorLostFailure (executor 2 exited caused by one of the running tasks) Reason: Command exited with code 137
26/08/10 14:22:04 WARN TaskSetManager: Lost task 2.1 in stage 1052.0 (TID 1146) (172.18.0.8 executor 3): FetchFailed(null, shuffleId=325, mapIndex=-1, mapId=-1, reduceId=126, message=
org.apache.spark.shuffle.MetadataFetchFailedException: Missing an output location for shuffle 325 partition 126
	at org.apache.spark.MapOutputTracker$.validateStatus(MapOutputTracker.scala:1739)
	at org.apache.spark.MapOutputTracker$.$anonfun$convertMapStatuses$11(MapOutputTracker.scala:1686)
	at org.apache.spark.MapOutputTracker$.$anonfun$convertMapStatuses$11$adapted(MapOutputTracker.scala:1685)
	at scala.collection.Iterator.foreach(Iterator.scala:943)
	at scala.collection.Iterator.foreach

### Installments_payments

In [13]:
df_inst = spark.read.csv("hdfs://namenode:8020/data/bronze/home_credit/raw/installments_payments.csv", header=True, inferSchema=True)
df_inst.createOrReplaceTempView("inst_raw")

spark.sql("""
    CREATE OR REPLACE TEMP VIEW inst_clean AS
    SELECT DISTINCT
        SK_ID_PREV,
        SK_ID_CURR,
        COALESCE(NUM_INSTALMENT_VERSION, 0) AS NUM_INSTALMENT_VERSION,
        NUM_INSTALMENT_NUMBER,
        COALESCE(DAYS_INSTALMENT, 0) AS DAYS_INSTALMENT,
        COALESCE(DAYS_ENTRY_PAYMENT, 0) AS DAYS_ENTRY_PAYMENT,
        COALESCE(AMT_INSTALMENT, 0) AS AMT_INSTALMENT,
        COALESCE(AMT_PAYMENT, 0) AS AMT_PAYMENT
    FROM inst_raw
""")

spark.table("inst_clean").write.mode("overwrite").parquet("hdfs://namenode:8020/data/silver/staging/installments_payments_clean")

### Credit_Card_Balence

In [14]:
df_cc = spark.read.csv("hdfs://namenode:8020/data/bronze/home_credit/raw/credit_card_balance.csv", header=True, inferSchema=True)
df_cc.createOrReplaceTempView("cc_raw")

spark.sql("""
    CREATE OR REPLACE TEMP VIEW cc_clean AS
    SELECT DISTINCT
        SK_ID_PREV,
        SK_ID_CURR,
        MONTHS_BALANCE,
        COALESCE(AMT_BALANCE, 0) AS AMT_BALANCE,
        COALESCE(AMT_CREDIT_LIMIT_ACTUAL, 0) AS AMT_CREDIT_LIMIT_ACTUAL,
        COALESCE(AMT_DRAWINGS_ATM_CURRENT, 0) AS AMT_DRAWINGS_ATM_CURRENT,
        COALESCE(AMT_DRAWINGS_CURRENT, 0) AS AMT_DRAWINGS_CURRENT,
        COALESCE(AMT_DRAWINGS_OTHER_CURRENT, 0) AS AMT_DRAWINGS_OTHER_CURRENT,
        COALESCE(AMT_DRAWINGS_POS_CURRENT, 0) AS AMT_DRAWINGS_POS_CURRENT,
        COALESCE(AMT_INST_MIN_REGULARITY, 0) AS AMT_INST_MIN_REGULARITY,
        COALESCE(AMT_PAYMENT_CURRENT, 0) AS AMT_PAYMENT_CURRENT,
        COALESCE(AMT_PAYMENT_TOTAL_CURRENT, 0) AS AMT_PAYMENT_TOTAL_CURRENT,
        COALESCE(AMT_RECEIVABLE_PRINCIPAL, 0) AS AMT_RECEIVABLE_PRINCIPAL,
        COALESCE(AMT_RECIVABLE, 0) AS AMT_RECIVABLE,
        COALESCE(AMT_TOTAL_RECEIVABLE, 0) AS AMT_TOTAL_RECEIVABLE,
        COALESCE(CNT_DRAWINGS_ATM_CURRENT, 0) AS CNT_DRAWINGS_ATM_CURRENT,
        COALESCE(CNT_DRAWINGS_CURRENT, 0) AS CNT_DRAWINGS_CURRENT,
        COALESCE(CNT_DRAWINGS_OTHER_CURRENT, 0) AS CNT_DRAWINGS_OTHER_CURRENT,
        COALESCE(CNT_DRAWINGS_POS_CURRENT, 0) AS CNT_DRAWINGS_POS_CURRENT,
        COALESCE(CNT_INSTALMENT_MATURE_CUM, 0) AS CNT_INSTALMENT_MATURE_CUM,
        COALESCE(NAME_CONTRACT_STATUS, 'Unknown') AS NAME_CONTRACT_STATUS,
        COALESCE(SK_DPD, 0) AS SK_DPD,
        COALESCE(SK_DPD_DEF, 0) AS SK_DPD_DEF
    FROM cc_raw
""")

spark.table("cc_clean").write.mode("overwrite").parquet("hdfs://namenode:8020/data/silver/staging/credit_card_balance_clean")
print("Credit Card Balance clean saved (SQL).")

Credit Card Balance clean saved (SQL).


### bureau_credit_features

In [15]:
# Baca bureau clean dari staging
df_bureau_clean = spark.read.parquet("hdfs://namenode:8020/data/silver/staging/bureau_clean")
df_bureau_clean.createOrReplaceTempView("bureau_clean")

# Buat fitur kredit biro per SK_ID_CURR
spark.sql("""
    CREATE OR REPLACE TEMP VIEW bureau_credit_features AS
    SELECT
        SK_ID_CURR,
        COUNT(SK_ID_BUREAU) AS BUREAU_CNT,
        SUM(CASE WHEN CREDIT_ACTIVE = 'Active' THEN 1 ELSE 0 END) AS BUREAU_ACTIVE_CNT,
        SUM(CASE WHEN CREDIT_ACTIVE = 'Closed' THEN 1 ELSE 0 END) AS BUREAU_CLOSED_CNT,
        AVG(ABS(DAYS_CREDIT) / 365.25) AS BUREAU_AVG_CREDIT_DURATION,
        MAX(AMT_CREDIT_MAX_OVERDUE) AS BUREAU_MAX_OVERDUE,
        AVG(AMT_CREDIT_MAX_OVERDUE) AS BUREAU_AVG_OVERDUE,
        SUM(AMT_CREDIT_SUM) AS BUREAU_TOTAL_CREDIT_SUM,
        SUM(AMT_CREDIT_SUM_DEBT) AS BUREAU_TOTAL_DEBT,
        CASE WHEN SUM(AMT_CREDIT_SUM) > 0 THEN SUM(AMT_CREDIT_SUM_DEBT) / SUM(AMT_CREDIT_SUM) ELSE 0 END AS BUREAU_DEBT_CREDIT_RATIO,
        AVG(AMT_ANNUITY) AS BUREAU_AVG_ANNUITY,
        SUM(CNT_CREDIT_PROLONG) AS BUREAU_TOTAL_PROLONG,
        MAX(DAYS_CREDIT_UPDATE) AS BUREAU_DAYS_SINCE_LAST_UPDATE
    FROM bureau_clean
    GROUP BY SK_ID_CURR
""")

# Lihat hasil
spark.table("bureau_credit_features").show(5, truncate=False)

+----------+----------+-----------------+-----------------+--------------------------+------------------+------------------+-----------------------+-----------------+------------------------+------------------+--------------------+-----------------------------+
|SK_ID_CURR|BUREAU_CNT|BUREAU_ACTIVE_CNT|BUREAU_CLOSED_CNT|BUREAU_AVG_CREDIT_DURATION|BUREAU_MAX_OVERDUE|BUREAU_AVG_OVERDUE|BUREAU_TOTAL_CREDIT_SUM|BUREAU_TOTAL_DEBT|BUREAU_DEBT_CREDIT_RATIO|BUREAU_AVG_ANNUITY|BUREAU_TOTAL_PROLONG|BUREAU_DAYS_SINCE_LAST_UPDATE|
+----------+----------+-----------------+-----------------+--------------------------+------------------+------------------+-----------------------+-----------------+------------------------+------------------+--------------------+-----------------------------+
|330299    |15        |3                |12               |4.5037644667              |14652.0           |3297.936          |1718253.0              |120973.5         |0.07040494036675624     |1593.738          |0   

In [16]:
bureau_credit_features = spark.table("bureau_credit_features")
print("Jumlah SK_ID_CURR unik di bureau_credit_features:", bureau_credit_features.count())

bureau_credit_features.write.mode("overwrite").parquet("hdfs://namenode:8020/data/silver/staging/bureau_credit_features")
print("bureau_credit_features saved.")

Jumlah SK_ID_CURR unik di bureau_credit_features: 305811


bureau_credit_features saved.


### bureau_delinquency_features

In [17]:
df_bb_clean = spark.read.parquet("hdfs://namenode:8020/data/silver/staging/bureau_balance_clean")
df_bb_clean.createOrReplaceTempView("bb_clean")

df_bureau_clean.createOrReplaceTempView("bureau_clean")  

spark.sql("""
    CREATE OR REPLACE TEMP VIEW bureau_delinquency_features AS
    SELECT
        b.SK_ID_CURR,
        AVG(CASE WHEN bb.STATUS IN ('C','X') THEN 0 ELSE CAST(bb.STATUS AS INT) END) AS BB_AVG_STATUS,
        MAX(CASE WHEN bb.STATUS IN ('C','X') THEN 0 ELSE CAST(bb.STATUS AS INT) END) AS BB_MAX_STATUS,
        SUM(CASE WHEN bb.STATUS NOT IN ('C','X') AND CAST(bb.STATUS AS INT) >= 1 THEN 1 ELSE 0 END) AS BB_OVERDUE_MONTHS,
        CASE WHEN COUNT(bb.MONTHS_BALANCE) > 0 
             THEN SUM(CASE WHEN bb.STATUS NOT IN ('C','X') AND CAST(bb.STATUS AS INT) >= 1 THEN 1 ELSE 0 END) * 1.0 / COUNT(bb.MONTHS_BALANCE) 
             ELSE 0 END AS BB_OVERDUE_RATIO,
        MAX(bb.MONTHS_BALANCE) - MIN(bb.MONTHS_BALANCE) AS BB_HISTORY_LENGTH
    FROM bb_clean bb
    JOIN bureau_clean b ON bb.SK_ID_BUREAU = b.SK_ID_BUREAU
    GROUP BY b.SK_ID_CURR
""")

spark.table("bureau_delinquency_features").show(5, truncate=False)
print("Jumlah bureau_delinquency_features:", spark.table("bureau_delinquency_features").count())
spark.table("bureau_delinquency_features").write.mode("overwrite").parquet("hdfs://namenode:8020/data/silver/staging/bureau_delinquency_features")
print("bureau_delinquency_features saved.")

+----------+--------------------+-------------+-----------------+------------------+-----------------+
|SK_ID_CURR|BB_AVG_STATUS       |BB_MAX_STATUS|BB_OVERDUE_MONTHS|BB_OVERDUE_RATIO  |BB_HISTORY_LENGTH|
+----------+--------------------+-------------+-----------------+------------------+-----------------+
|330299    |0.012224938875305624|1            |10               |0.0122249388753056|86               |
|135976    |0.007009345794392523|1            |3                |0.0070093457943925|58               |
|281607    |0.0                 |0            |0                |0.0000000000000000|53               |
|126191    |0.0                 |0            |0                |0.0000000000000000|50               |
|260195    |0.003389830508474576|1            |1                |0.0033898305084746|87               |
+----------+--------------------+-------------+-----------------+------------------+-----------------+
only showing top 5 rows



26/08/10 14:30:00 ERROR TaskSchedulerImpl: Lost executor 3 on 172.18.0.8: Command exited with code 137
26/08/10 14:30:00 WARN TaskSetManager: Lost task 1.0 in stage 1081.0 (TID 1205) (172.18.0.8 executor 3): ExecutorLostFailure (executor 3 exited caused by one of the running tasks) Reason: Command exited with code 137
26/08/10 14:30:00 WARN TaskSetManager: Lost task 0.0 in stage 1081.0 (TID 1204) (172.18.0.8 executor 3): ExecutorLostFailure (executor 3 exited caused by one of the running tasks) Reason: Command exited with code 137


Jumlah bureau_delinquency_features: 134542


bureau_delinquency_features saved.


#### previous_application_features

In [ ]:
df_prev_clean = spark.read.parquet("hdfs://namenode:8020/data/silver/staging/previous_application_clean")
df_prev_clean.createOrReplaceTempView("prev_clean")

spark.sql("""
    CREATE OR REPLACE TEMP VIEW previous_application_features AS
    SELECT
        SK_ID_CURR,
        COUNT(SK_ID_PREV) AS PREV_CNT,
        SUM(CASE WHEN NAME_CONTRACT_STATUS = 'Approved' THEN 1 ELSE 0 END) AS PREV_APPROVED_CNT,
        SUM(CASE WHEN NAME_CONTRACT_STATUS = 'Refused' THEN 1 ELSE 0 END) AS PREV_REFUSED_CNT,
        CASE WHEN COUNT(SK_ID_PREV) > 0 THEN SUM(CASE WHEN NAME_CONTRACT_STATUS = 'Approved' THEN 1 ELSE 0 END) * 1.0 / COUNT(SK_ID_PREV) ELSE 0 END AS PREV_APPROVAL_RATE,
        AVG(AMT_CREDIT) AS PREV_AVG_AMT_CREDIT,
        SUM(AMT_CREDIT) AS PREV_TOTAL_AMT_CREDIT,
        AVG(AMT_DOWN_PAYMENT) AS PREV_AVG_DOWN_PAYMENT,
        AVG(RATE_INTEREST_PRIMARY) AS PREV_AVG_INTEREST,
        SUM(CASE WHEN NAME_CONTRACT_TYPE LIKE '%Cash%' THEN 1 ELSE 0 END) AS PREV_CASH_LOAN_CNT,
        SUM(CASE WHEN NAME_CONTRACT_TYPE LIKE '%Revolving%' THEN 1 ELSE 0 END) AS PREV_REVOLVING_CNT,
        MAX(DAYS_DECISION) AS PREV_DAYS_SINCE_LAST_APP,
        SUM(NFLAG_INSURED_ON_APPROVAL) AS PREV_INSURED_CNT
    FROM prev_clean
    GROUP BY SK_ID_CURR
""")

spark.table("previous_application_features").show(5, truncate=False)
print("Jumlah previous_application_features:", spark.table("previous_application_features").count())
spark.table("previous_application_features").write.mode("overwrite").parquet("hdfs://namenode:8020/data/silver/staging/previous_application_features")
print("previous_application_features saved.")

#### pos_loan_features

In [ ]:
df_pos_clean = spark.read.parquet("hdfs://namenode:8020/data/silver/staging/POS_CASH_balance_clean")
df_pos_clean.createOrReplaceTempView("pos_clean")

spark.sql("""
    CREATE OR REPLACE TEMP VIEW pos_loan_features AS
    SELECT
        SK_ID_CURR,
        COUNT(DISTINCT SK_ID_PREV) AS POS_CNT,
        AVG(CNT_INSTALMENT_FUTURE) AS POS_AVG_REMAINING_INST,
        AVG(SK_DPD) AS POS_AVG_DPD,
        MAX(SK_DPD) AS POS_MAX_DPD,
        SUM(CASE WHEN NAME_CONTRACT_STATUS = 'Active' THEN 1 ELSE 0 END) AS POS_ACTIVE_CNT,
        SUM(CASE WHEN NAME_CONTRACT_STATUS = 'Completed' THEN 1 ELSE 0 END) AS POS_COMPLETED_CNT,
        CASE WHEN COUNT(*) > 0 THEN SUM(CASE WHEN SK_DPD > 0 THEN 1 ELSE 0 END) * 1.0 / COUNT(*) ELSE 0 END AS POS_OVERDUE_RATIO
    FROM pos_clean
    GROUP BY SK_ID_CURR
""")

# Simpan
spark.table("pos_loan_features").write.mode("overwrite").parquet("hdfs://namenode:8020/data/silver/staging/pos_loan_features")
print("pos_loan_features saved.")

#### installment_payment_features

In [ ]:
df_inst_clean = spark.read.parquet("hdfs://namenode:8020/data/silver/staging/installments_payments_clean")
df_inst_clean.createOrReplaceTempView("inst_clean")

spark.sql("""
    CREATE OR REPLACE TEMP VIEW installment_payment_features AS
    SELECT
        SK_ID_CURR,
        COUNT(*) AS INSTAL_CNT,
        AVG(AMT_PAYMENT / NULLIF(AMT_INSTALMENT, 0)) AS INSTAL_AVG_PAYMENT_RATIO,
        STDDEV(AMT_PAYMENT / NULLIF(AMT_INSTALMENT, 0)) AS INSTAL_STD_PAYMENT_RATIO,
        SUM(CASE WHEN DAYS_ENTRY_PAYMENT > DAYS_INSTALMENT THEN 1 ELSE 0 END) AS INSTAL_LATE_CNT,
        CASE WHEN COUNT(*) > 0 THEN SUM(CASE WHEN DAYS_ENTRY_PAYMENT > DAYS_INSTALMENT THEN 1 ELSE 0 END) * 1.0 / COUNT(*) ELSE 0 END AS INSTAL_LATE_RATIO,
        AVG(CASE WHEN DAYS_ENTRY_PAYMENT > DAYS_INSTALMENT THEN DAYS_ENTRY_PAYMENT - DAYS_INSTALMENT ELSE 0 END) AS INSTAL_AVG_DAYS_LATE,
        SUM(AMT_PAYMENT) AS INSTAL_TOTAL_PAYMENT
    FROM inst_clean
    GROUP BY SK_ID_CURR
""")

spark.table("installment_payment_features").write.mode("overwrite").parquet("hdfs://namenode:8020/data/silver/staging/installment_payment_features")
print("installment_payment_features saved.")

#### credit_card_features

In [ ]:
df_cc_clean = spark.read.parquet("hdfs://namenode:8020/data/silver/staging/credit_card_balance_clean")
df_cc_clean.createOrReplaceTempView("cc_clean")

spark.sql("""
    CREATE OR REPLACE TEMP VIEW credit_card_features AS
    SELECT
        SK_ID_CURR,
        COUNT(DISTINCT SK_ID_PREV) AS CC_CNT,
        AVG(AMT_BALANCE) AS CC_AVG_BALANCE,
        SUM(AMT_CREDIT_LIMIT_ACTUAL) AS CC_TOTAL_LIMIT,
        AVG(AMT_BALANCE / NULLIF(AMT_CREDIT_LIMIT_ACTUAL, 0)) AS CC_AVG_UTILIZATION,
        MAX(AMT_BALANCE / NULLIF(AMT_CREDIT_LIMIT_ACTUAL, 0)) AS CC_MAX_UTILIZATION,
        AVG(AMT_PAYMENT_TOTAL_CURRENT) AS CC_AVG_PAYMENT,
        AVG(AMT_DRAWINGS_ATM_CURRENT) AS CC_AVG_ATM_DRAWINGS,
        CASE WHEN COUNT(*) > 0 THEN SUM(CASE WHEN AMT_BALANCE > AMT_CREDIT_LIMIT_ACTUAL THEN 1 ELSE 0 END) * 1.0 / COUNT(*) ELSE 0 END AS CC_OVERLIMIT_RATIO,
        AVG(SK_DPD) AS CC_AVG_DPD,
        MAX(SK_DPD) AS CC_MAX_DPD
    FROM cc_clean
    GROUP BY SK_ID_CURR
""")

spark.table("credit_card_features").write.mode("overwrite").parquet("hdfs://namenode:8020/data/silver/staging/credit_card_features")
print("credit_card_features saved.")

#### Main Table A

In [ ]:
# Baca tabel utama train yang sudah bersih
df_train_clean = spark.read.parquet("hdfs://namenode:8020/data/silver/staging/application_train_clean")
df_train_clean.createOrReplaceTempView("t")

# Baca semua fitur agregat dengan alias pendek
spark.read.parquet("hdfs://namenode:8020/data/silver/staging/bureau_credit_features").createOrReplaceTempView("a")
spark.read.parquet("hdfs://namenode:8020/data/silver/staging/bureau_delinquency_features").createOrReplaceTempView("b")
spark.read.parquet("hdfs://namenode:8020/data/silver/staging/previous_application_features").createOrReplaceTempView("c")
spark.read.parquet("hdfs://namenode:8020/data/silver/staging/pos_loan_features").createOrReplaceTempView("d")
spark.read.parquet("hdfs://namenode:8020/data/silver/staging/installment_payment_features").createOrReplaceTempView("e")
spark.read.parquet("hdfs://namenode:8020/data/silver/staging/credit_card_features").createOrReplaceTempView("f")

# Lakukan left join semua fitur ke tabel utama
spark.sql("""
    CREATE OR REPLACE TEMP VIEW train_integrated AS
    SELECT
        t.*,
        a.BUREAU_CNT, a.BUREAU_ACTIVE_CNT, a.BUREAU_CLOSED_CNT,
        a.BUREAU_AVG_CREDIT_DURATION, a.BUREAU_MAX_OVERDUE, a.BUREAU_AVG_OVERDUE,
        a.BUREAU_TOTAL_CREDIT_SUM, a.BUREAU_TOTAL_DEBT, a.BUREAU_DEBT_CREDIT_RATIO,
        a.BUREAU_AVG_ANNUITY, a.BUREAU_TOTAL_PROLONG, a.BUREAU_DAYS_SINCE_LAST_UPDATE,
        b.BB_AVG_STATUS, b.BB_MAX_STATUS, b.BB_OVERDUE_MONTHS,
        b.BB_OVERDUE_RATIO, b.BB_HISTORY_LENGTH,
        c.PREV_CNT, c.PREV_APPROVED_CNT, c.PREV_REFUSED_CNT,
        c.PREV_APPROVAL_RATE, c.PREV_AVG_AMT_CREDIT, c.PREV_TOTAL_AMT_CREDIT,
        c.PREV_AVG_DOWN_PAYMENT, c.PREV_AVG_INTEREST,
        c.PREV_CASH_LOAN_CNT, c.PREV_REVOLVING_CNT,
        c.PREV_DAYS_SINCE_LAST_APP, c.PREV_INSURED_CNT,
        d.POS_CNT, d.POS_AVG_REMAINING_INST, d.POS_AVG_DPD, d.POS_MAX_DPD,
        d.POS_ACTIVE_CNT, d.POS_COMPLETED_CNT, d.POS_OVERDUE_RATIO,
        e.INSTAL_CNT, e.INSTAL_AVG_PAYMENT_RATIO, e.INSTAL_STD_PAYMENT_RATIO,
        e.INSTAL_LATE_CNT, e.INSTAL_LATE_RATIO, e.INSTAL_AVG_DAYS_LATE,
        e.INSTAL_TOTAL_PAYMENT,
        f.CC_CNT, f.CC_AVG_BALANCE, f.CC_TOTAL_LIMIT,
        f.CC_AVG_UTILIZATION, f.CC_MAX_UTILIZATION,
        f.CC_AVG_PAYMENT, f.CC_AVG_ATM_DRAWINGS,
        f.CC_OVERLIMIT_RATIO, f.CC_AVG_DPD, f.CC_MAX_DPD
    FROM t
    LEFT JOIN a ON t.SK_ID_CURR = a.SK_ID_CURR
    LEFT JOIN b ON t.SK_ID_CURR = b.SK_ID_CURR
    LEFT JOIN c ON t.SK_ID_CURR = c.SK_ID_CURR
    LEFT JOIN d ON t.SK_ID_CURR = d.SK_ID_CURR
    LEFT JOIN e ON t.SK_ID_CURR = e.SK_ID_CURR
    LEFT JOIN f ON t.SK_ID_CURR = f.SK_ID_CURR
""")

# Simpan tabel terintegrasi
spark.table("train_integrated") \
    .coalesce(2) \
    .write.mode("overwrite") \
    .parquet("hdfs://namenode:8020/data/silver/staging/train_integrated")

# Verifikasi
print("Train integrated count:", spark.table("train_integrated").count())
spark.table("train_integrated").select("SK_ID_CURR", "TARGET", "BUREAU_CNT", "PREV_CNT", "CC_CNT").show(5)

#### Main Table B

In [ ]:
df_train_clean = spark.read.parquet("hdfs://namenode:8020/data/silver/staging/application_train_clean")
df_train_clean.createOrReplaceTempView("train_clean")

spark.sql("""
    CREATE OR REPLACE TEMP VIEW train_integrated AS
    SELECT
        t.*,
        a.BUREAU_CNT, a.BUREAU_ACTIVE_CNT, a.BUREAU_CLOSED_CNT,
        a.BUREAU_AVG_CREDIT_DURATION, a.BUREAU_MAX_OVERDUE, a.BUREAU_AVG_OVERDUE,
        a.BUREAU_TOTAL_CREDIT_SUM, a.BUREAU_TOTAL_DEBT, a.BUREAU_DEBT_CREDIT_RATIO,
        a.BUREAU_AVG_ANNUITY, a.BUREAU_TOTAL_PROLONG, a.BUREAU_DAYS_SINCE_LAST_UPDATE,
        b.BB_AVG_STATUS, b.BB_MAX_STATUS, b.BB_OVERDUE_MONTHS,
        b.BB_OVERDUE_RATIO, b.BB_HISTORY_LENGTH,
        c.PREV_CNT, c.PREV_APPROVED_CNT, c.PREV_REFUSED_CNT,
        c.PREV_APPROVAL_RATE, c.PREV_AVG_AMT_CREDIT, c.PREV_TOTAL_AMT_CREDIT,
        c.PREV_AVG_DOWN_PAYMENT, c.PREV_AVG_INTEREST,
        c.PREV_CASH_LOAN_CNT, c.PREV_REVOLVING_CNT,
        c.PREV_DAYS_SINCE_LAST_APP, c.PREV_INSURED_CNT,
        d.POS_CNT, d.POS_AVG_REMAINING_INST, d.POS_AVG_DPD, d.POS_MAX_DPD,
        d.POS_ACTIVE_CNT, d.POS_COMPLETED_CNT, d.POS_OVERDUE_RATIO,
        e.INSTAL_CNT, e.INSTAL_AVG_PAYMENT_RATIO, e.INSTAL_STD_PAYMENT_RATIO,
        e.INSTAL_LATE_CNT, e.INSTAL_LATE_RATIO, e.INSTAL_AVG_DAYS_LATE,
        e.INSTAL_TOTAL_PAYMENT,
        f.CC_CNT, f.CC_AVG_BALANCE, f.CC_TOTAL_LIMIT,
        f.CC_AVG_UTILIZATION, f.CC_MAX_UTILIZATION,
        f.CC_AVG_PAYMENT, f.CC_AVG_ATM_DRAWINGS,
        f.CC_OVERLIMIT_RATIO, f.CC_AVG_DPD, f.CC_MAX_DPD
    FROM train_clean t
    LEFT JOIN bureau_credit_features a ON t.SK_ID_CURR = a.SK_ID_CURR
    LEFT JOIN bureau_delinquency_features b ON t.SK_ID_CURR = b.SK_ID_CURR
    LEFT JOIN previous_application_features c ON t.SK_ID_CURR = c.SK_ID_CURR
    LEFT JOIN pos_loan_features d ON t.SK_ID_CURR = d.SK_ID_CURR
    LEFT JOIN installment_payment_features e ON t.SK_ID_CURR = e.SK_ID_CURR
    LEFT JOIN credit_card_features f ON t.SK_ID_CURR = f.SK_ID_CURR
""")

spark.table("train_integrated") \
    .coalesce(2) \
    .write.mode("overwrite") \
    .parquet("hdfs://namenode:8020/data/silver/staging/train_integrated")

print("Train integrated count:", spark.table("train_integrated").count())

In [ ]:
# Validasi Train
train = spark.read.parquet("hdfs://namenode:8020/data/silver/staging/train_integrated")

# 1. SK_ID_CURR not null
null_sk = train.filter(col("SK_ID_CURR").isNull()).count()
print(f"SK_ID_CURR null: {null_sk} (harus 0)")

# 2. SK_ID_CURR unique
total = train.count()
unique = train.select("SK_ID_CURR").distinct().count()
print(f"Total: {total}, Unique: {unique}, Duplikat: {total - unique} (harus 0)")

# 3. TARGET in (0,1)
invalid_target = train.filter(~col("TARGET").isin([0, 1])).count()
print(f"TARGET invalid: {invalid_target} (harus 0)")

# 4. AGE_YEARS between 18-100
outlier_age = train.filter((col("AGE_YEARS") < 18) | (col("AGE_YEARS") > 100)).count()
print(f"AGE out of range: {outlier_age} (harus 0)")

print("\n=== Hasil Validasi ===")
if null_sk == 0 and total == unique and invalid_target == 0 and outlier_age == 0:
    print("✅ Semua validasi PASS")
else:
    print("❌ Ada validasi GAGAL, periksa kembali")

In [ ]:
# Baca ulang tabel yang sudah divalidasi
train_valid = spark.read.parquet("hdfs://namenode:8020/data/silver/staging/train_integrated")
test_valid = spark.read.parquet("hdfs://namenode:8020/data/silver/staging/test_integrated")

# Simpan ke folder final
train_valid.coalesce(2).write.mode("overwrite").parquet("hdfs://namenode:8020/data/silver/home_credit/integrated/train")
test_valid.coalesce(2).write.mode("overwrite").parquet("hdfs://namenode:8020/data/silver/home_credit/integrated/test")

print("Data final train & test disimpan di /data/silver/home_credit/integrated/")